# CR-Zero: Offline Behavior Cloning Training Pipeline

This notebook trains the **Deck-Conditioned Policy Network** for the **CR-Zero Clash Royale AI Agent** using offline imitation learning (Behavior Cloning).

### Features:
- **Dual-environment support**: Seamlessly runs locally inside your IDE (VS Code / Antigravity with local GPU) or in Google Colab (free T4 / A100 GPU).
- **Global 135-card vocabulary**: Learned embeddings combined with static attribute projections (cost, range, speed, target).
- **Outcome-weighted loss**: Trajectories from winning matches ($1.5\times$) are prioritized over losses ($0.5\times$).
- **Direct model export**: Automatically outputs `checkpoints/cr_zero_policy_best.pt` and `cr_zero_policy_best.onnx`.

In [ ]:
# Cell 1: Environment & Compute Device Setup
import os
import sys

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("[*] Running in Google Colab Environment")
    !apt-get update -qq && apt-get install -y -qq ffmpeg git
    !pip install -q yt-dlp ultralytics onnx onnxruntime-gpu opencv-python
    if not os.path.exists("clash-ml-model"):
        !git clone https://github.com/varun-zero/clash-ml-model.git
        %cd clash-ml-model
else:
    print("[*] Running in Local IDE Environment")

# Ensure project root is in sys.path
repo_root = os.path.abspath(os.getcwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import torch
print(f"PyTorch:         {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU:      {torch.cuda.get_device_name(0)}")


## 1. Inspect Trajectory Dataset Shards

Check that match trajectory shards (`.npz`) exist in `data/trajectories/` and verify their spatial grids, scalars, and legal masks.

In [ ]:
# Cell 2: Dataset Verification
import glob
import numpy as np

data_dir = os.path.join(".", "data", "trajectories")
shards = sorted(glob.glob(os.path.join(data_dir, "*.npz")))
print(f"Found {len(shards)} trajectory shard(s) in {data_dir}:")

for s in shards:
    data = np.load(s)
    outcome = int(data["match_outcome"]) if "match_outcome" in data else 0
    outcome_str = "WIN (+1)" if outcome == 1 else ("LOSS (-1)" if outcome == -1 else "DRAW (0)")
    print(f"  -> {os.path.basename(s)}: {data['arena_grid'].shape[0]} frames | Outcome: {outcome_str} | Grid: {data['arena_grid'].shape}")


## 2. Train Behavior Cloning Policy Network

We train the model using mixed precision, Cosine Annealing learning rate schedule, and AdamW optimizer. Val loss and accuracy metrics are logged each epoch.

In [ ]:
# Cell 3: Execute Training Loop
!python colab_pipeline/train_bc.py \
    --data_dir ./data/trajectories \
    --save_dir ./checkpoints \
    --epochs 10 \
    --batch_size 32 \
    --lr 1e-3


## 3. Verify Saved Checkpoints & Model Inference

Load the newly trained `cr_zero_policy_best.pt` checkpoint and verify a single inference pass.

In [ ]:
# Cell 4: Test Model Inference Pass
import torch
from policy.model import DeckConditionedPolicy
from configs.constants import NUM_TOTAL_CARDS, STACKED_GRID_CHANNELS

checkpoint_path = "./checkpoints/cr_zero_policy_best.pt"
if os.path.exists(checkpoint_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    policy = DeckConditionedPolicy(num_card_classes=NUM_TOTAL_CARDS, in_channels=STACKED_GRID_CHANNELS).to(device)
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=True)
    policy.load_state_dict(ckpt["model_state_dict"])
    policy.eval()
    
    # Run a test forward pass
    dummy_grid = torch.randn(1, 32, 32, 18, device=device)
    dummy_scalars = torch.randn(1, 12, device=device)
    dummy_hand = torch.tensor([[1, 2, 20, 21]], dtype=torch.long, device=device)
    dummy_cmask = torch.tensor([[True, True, True, True, True]], dtype=torch.bool, device=device)
    dummy_tmask = torch.ones(1, 576, dtype=torch.bool, device=device)
    
    with torch.no_grad():
        card_slot, tile_idx, card_p, tile_p = policy.sample_action(
            dummy_grid, dummy_scalars, dummy_hand, dummy_cmask, dummy_tmask, temperature=0.7
        )
    print(f"[OK] Checkpoint verified! Inference sample -> Card Slot: {card_slot}, Tile Index: {tile_idx}")
    
    if IS_COLAB:
        from google.colab import files
        files.download(checkpoint_path)
        if os.path.exists("./checkpoints/cr_zero_policy_best.onnx"):
            files.download("./checkpoints/cr_zero_policy_best.onnx")
else:
    print(f"Checkpoint {checkpoint_path} not found. Run Cell 3 first.")
